# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
1. **What one row means:** One row in the source fact table is one content item for one client on one report_date (a daily content-performance observation).

2. **Time window:** I use March 2026 as the feature/development month. The feature values are knowable by the end of March; the outcome window is April 2026, which is kept separate from the features.

3. **What I predict/rank:** I predict whether a content item will have an impression decline of more than 20% in April versus March. The label is 1 when (April impressions - March impressions) / March impressions < -0.20, otherwise 0. I would use the score to rank content items for review.

4. **Tables:** I use fact_content_daily_performance for daily performance and its March/April partitions. I use no client names and no query-level table for this small contract.

5. **Deliberate exclusion:** I deliberately exclude April performance from the feature set because it is the future outcome window; I also exclude pseudonymous IDs from model features because they are identifiers, not signals.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Setup: DuckDB + Hugging Face authentication.
# The token is read from the Colab Secret HF_TOKEN. It is never written into this notebook.

%pip -q install duckdb huggingface_hub scikit-learn pandas

import os
import duckdb
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. In Colab: open the key/Secrets panel, add a Secret named HF_TOKEN, "
        "then Run all again."
    )

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

print("Connected. Development slice: March 2026; outcome window: April 2026.")

Connected. Development slice: March 2026; outcome window: April 2026.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature (safe at the decision moment):**
- March `gsc_impressions` — search demand observed during March.
- March `gsc_clicks` — search clicks observed during March.
- March mean `gsc_avg_position` — average search position observed during March.
- March `impression_days` — number of March days with at least one impression.
- March `active_days_span` — number of calendar days between the first and last March observation for the item.

**Label / proxy:**
- `april_decline_label` — 1 if April impressions are more than 20% below March impressions, otherwise 0.
- `april_trend_pct` — the percentage change used to derive the label. It is a label source, never a feature.

**Context:**
- `client_hash_id`, `content_hash_id`, and `report_date` — needed for grouping, joining, and checking the panel, not for model learning.
- Month/partition information — used to control the time window.

**Excluded:**
- April performance columns — future information unavailable at the March decision moment.
- `april_trend_pct` — deliberately excluded after the leakage experiment because it is the label in numeric disguise.
- Query-level features — excluded from this small contract because their fixed 90-day window can overlap the outcome period and would require an additional window-alignment check.
- Client/content IDs — excluded from model features because they are identifiers.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell only states the intended feature schema; the three verification queries are below.
FEATURES = [
    "march_impressions",
    "march_clicks",
    "march_mean_position",
    "march_impression_days",
    "march_active_days_span",
]
print("Final honest feature list:", FEATURES)


Final honest feature list: ['march_impressions', 'march_clicks', 'march_mean_position', 'march_impression_days', 'march_active_days_span']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The three checks below are deliberately small:

1. **Grain:** duplicate `(report_date, client_hash_id, content_hash_id)` keys should be absent.
2. **Slice count + date span:** March 2026 row count and its observed date range.
3. **Availability:** GA4 availability is checked with `IS TRUE`, as required for the warehouse's three-valued flags.

The feature frame and leakage experiment come after these checks.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verification query 1 — grain
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
    FROM {FACT_MARCH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate grain keys (expected: 0 rows):")
display(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain keys (expected: 0 rows):


,report_date,client_hash_id,content_hash_id,row_count


In [12]:
# Verification query 2 — March slice row count + date span
slice_stats = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM {FACT_MARCH}
""").df()

display(slice_stats)


,march_rows,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


In [13]:
# Verification query 3 — availability, explicitly using IS TRUE
availability_stats = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS not_true_rows
    FROM {FACT_MARCH}
""").df()

display(availability_stats)
print("Only rows with ga4_data_available IS TRUE are considered GA4-available.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,ga4_available_rows,not_true_rows
0,9841378,413966,9427412


Only rows with ga4_data_available IS TRUE are considered GA4-available.


### Five-feature frame

Each feature is available **at the end of March, before the April outcome window**:

1. `march_impressions` — knowable at the decision moment because March search impressions have already been observed.
2. `march_clicks` — knowable at the decision moment because March search clicks have already been observed.
3. `march_mean_position` — knowable at the decision moment because March GSC positions have already been observed.
4. `march_impression_days` — knowable at the decision moment because it counts March days with impressions only.
5. `march_active_days_span` — knowable at the decision moment because it uses the first and last March observations only.

The frame below is built from March features and an April outcome label. Items with zero March impressions are excluded because a percentage decline from zero is not defined.


In [14]:
# Build the small March feature frame and April label.
# This is NOT one of the three verification queries; it is the modeling frame.

feature_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(NULLIF(gsc_avg_position, 0)) AS march_mean_position,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS march_impression_days,
        DATE_DIFF('day', MIN(report_date), MAX(report_date)) + 1 AS march_active_days_span
    FROM {FACT_MARCH}
    GROUP BY 1, 2
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM {FACT_APRIL}
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    m.march_clicks,
    m.march_mean_position,
    m.march_impression_days,
    m.march_active_days_span,
    a.april_impressions,
    100.0 * (a.april_impressions - m.march_impressions)
        / NULLIF(m.march_impressions, 0) AS april_trend_pct
FROM march m
JOIN april a
  ON m.client_hash_id = a.client_hash_id
 AND m.content_hash_id = a.content_hash_id
WHERE m.march_impressions > 0
"""

feature_frame = con.sql(feature_sql).df()
feature_frame["april_decline_label"] = (feature_frame["april_trend_pct"] < -20.0).astype(int)

print(f"Feature frame: {len(feature_frame):,} content items")
display(feature_frame[FEATURES + ["april_decline_label"]].head())
print("Decline rate:", round(feature_frame["april_decline_label"].mean(), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 176,737 content items


,march_impressions,march_clicks,march_mean_position,march_impression_days,march_active_days_span,april_decline_label
0,747.0,1.0,29.377302,31,31,1
1,501.0,2.0,17.519486,31,31,1
2,2893.0,1.0,9.445005,31,31,1
3,8.0,0.0,24.000000,6,31,0
4,6955.0,9.0,4.365083,31,31,1


Decline rate: 0.532


## 3b) The trap: deliberate leakage experiment

`april_trend_pct` is derived directly from the April outcome and the label is defined from it. I add it **on purpose** to the feature matrix, fit a quick classifier, and show the score. Then I delete it and report the honest score using only the five March features.

The point is the same as notebook 02: the leaked feature is the answer in disguise.

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

model_df = feature_frame.dropna(subset=["april_decline_label"]).copy()
y = model_df["april_decline_label"].astype(int)

X_honest = model_df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
X_leaky = X_honest.copy()
X_leaky["april_trend_pct"] = model_df["april_trend_pct"].fillna(0)

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_honest, y, test_size=0.25, random_state=42, stratify=y
)
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky, y, test_size=0.25, random_state=42, stratify=y
)

honest_model = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
leaky_model = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)

honest_model.fit(Xh_train, yh_train)
leaky_model.fit(Xl_train, yl_train)

honest_auc = roc_auc_score(yh_test, honest_model.predict_proba(Xh_test)[:, 1])
leaky_auc = roc_auc_score(yl_test, leaky_model.predict_proba(Xl_test)[:, 1])

print(f"Leaky AUC (with april_trend_pct): {leaky_auc:.3f}")
print(f"Honest AUC (five March features only): {honest_auc:.3f}")
print("Deleted from final feature frame:", "april_trend_pct")


Leaky AUC (with april_trend_pct): 1.000
Honest AUC (five March features only): 0.598
Deleted from final feature frame: april_trend_pct


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** this is an **unbalanced panel**: clients do not all have the same amount of historical data. A March row therefore does not imply the same depth of history for every client. This slice is also a development window, not a final out-of-time test; June 2026 remains sealed for later evaluation.

Other deliberate limits: the contract does not claim causality, and the next-month decline label only measures the observed impression change. It does not explain *why* a page declined.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final contract snapshot / self-check output
print("FINAL CONTRACT")
print("Grain: client × content × report_date daily performance row")
print("Feature month: 2026-03")
print("Outcome month: 2026-04")
print("Label: April impressions decline >20% vs March")
print("Honest features:", FEATURES)
print("Leaky field removed: april_trend_pct")
print("Sealed test month: 2026-06")


FINAL CONTRACT
Grain: client × content × report_date daily performance row
Feature month: 2026-03
Outcome month: 2026-04
Label: April impressions decline >20% vs March
Honest features: ['march_impressions', 'march_clicks', 'march_mean_position', 'march_impression_days', 'march_active_days_span']
Leaky field removed: april_trend_pct
Sealed test month: 2026-06


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.